Some of the content here is adapted from [LangChain Handbook](https://github.com/pinecone-io/examples/tree/master/generation/langchain/handbook)

# Agents 🤖

Agents are developed on top of LLMs to extend their capabilities. For example, agents provide tools to allow a LLM to access Google search, perform complex calculations with Python, and even make SQL queries.

**Definition**: The key behind agents is giving LLM's the possibility of using tools in their workflow. This is where langchain departs from the popular chatgpt implementation and we can start to get a glimpse of what it offers us as builders. Until now, we covered several building blocks in isolation. Let's see them come to life.

The official definition of agents is the following:


> Agents use an LLM to determine which actions to take and in what order. An action can either be using a tool and observing its output, or returning to the user.

In [1]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

In [2]:
llm = ChatOpenAI(model="gpt-6-luna")

In [7]:
# try to accomplish a task without agents and tools
res = llm.invoke("Given the sentence 'Hani is the best' reverse its words, then reverse the characters of each word.")

In [8]:
print(res.content)

tseb eht si inaH


In [9]:
@tool
def word_reverse(text: str) -> str:
    """Reverse the order of the words in a sentence."""
    return " ".join(text.split()[::-1])

@tool
def character_reverse(text: str) -> str:
    """Reverse the characters in a string."""
    return text[::-1]

In [10]:
agent1 = create_agent(
    model=llm,
    tools=[word_reverse, character_reverse],
    system_prompt="You are a helpful assistant. Use tools when needed.",
)

In [11]:
prompt = "Given the sentence 'Hani is the best' reverse its words, then reverse the characters of each word."
result = agent1.invoke({"messages": [{"role": "user", "content": prompt}]})

In [12]:
for m in result['messages']:
    print(m)
    print()

content="Given the sentence 'Hani is the best' reverse its words, then reverse the characters of each word." additional_kwargs={} response_metadata={} id='7823f198-1fb1-41e0-8430-ae6ccaf69315'

content=[{'arguments': '{"text":"Hani is the best"}', 'call_id': 'call_Z0Oo0YtAkvdfWc26n4KUuXAG', 'name': 'word_reverse', 'type': 'function_call', 'id': 'fc_07f2bea157ef0280006aee1e213e8887d194d92b33391f55c1', 'status': 'completed'}] additional_kwargs={} response_metadata={'id': 'resp_07f2bea157ef0280006aee1e20348087d1ac00f7e7934c7b94', 'created_at': 1793990176.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'} id='resp_07f2bea157ef0280006aee1e20348087d1ac00f7e7934c7b94' tool_calls=[{'name': 'word_reverse', 'args': {'text': 'Hani is the best'}, 'id': 'call_Z0Oo0YtAkvdfWc26n4KUuXAG', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 104, 'output_tokens':

## Example 2: tax assistant agent

In [13]:
def calculate_tax_irs(income):
    # Define the tax brackets and rates
    brackets = [(0, 20550, 0.10), (20551, 83550, 0.12), (83551, 178150, 0.22),
                (178151, 340100, 0.24), (340101, 431900, 0.32),
                (431901, 647850, 0.35), (647851, float('inf'), 0.37)]
    
    tax = 0
    for lower, upper, rate in brackets:
        if income > upper:
            tax += (upper - lower) * rate
        else:
            tax += (income - lower) * rate
            break

    return tax

In [14]:
@tool
def tax_resident(income: float) -> float:
    """Calculate the tax liability of residents. Input: income as a number."""
    print("-----> CALCULATING TAX RESIDENT ON INCOME", income)
    return calculate_tax_irs(income)

@tool
def tax_nonresident(income: float) -> float:
    """Calculate the tax liability of nonresidents. Input: income as a number."""
    print("-----> CALCULATING TAX NON RESIDENT ON INCOME", income)
    return income * 0.25

In [15]:
# https://www.irs.gov/individuals/international-taxpayers/determining-an-individuals-tax-residency-status
AGENT_PROMPT = """You are a tax assistant helping your creator to estimate the tax liability of employees.

First determine the residency status of the client. Below is the IRS guidelines.

'If you are not a U.S. citizen, you are considered a nonresident of the United States for U.S. tax purposes unless you meet one of two tests. You are a resident  of the United States for tax purposes if you meet either the green card test or the substantial presence test for the calendar year (January 1 – December 31).'

Once you determine the residency status, use your tools to calculate the tax liability depending on whether the client is a resident or nonresident.

"""

In [16]:
tax_agent = create_agent(
    model=llm,
    tools=[tax_resident, tax_nonresident],
    system_prompt=AGENT_PROMPT,
)

In [17]:
def ask_tax(client: str):
    result = tax_agent.invoke({"messages": [{"role": "user", 
                                             "content": "Client Information: "+client}]})
    return result["messages"][-1].content


In [18]:
print(ask_tax("John Maden made $100,000 he is US citizen"))

-----> CALCULATING TAX RESIDENT ON INCOME 100000.0
[{'type': 'text', 'text': 'John is a U.S. citizen, so he is a U.S. tax resident. For $100,000 of income, the estimated tax liability is **$13,233.66**.', 'annotations': [], 'id': 'msg_06025ffccbeb150f006aee1e32a26887d19f8d79431116d2dd', 'phase': 'final_answer'}]


In [19]:
print(ask_tax("John Maden is a foreign worker who worked three months as a contractor and made $100,000"))

[{'id': 'rs_03d0bb893d4feca5006aee1e3902dc87d1bd558308b2ae7973', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABq7h47GvYvmCGvuuPKzAUs9UvFhQe5Ee_Pf7YA_qiZsCydUO4Ek0TMVb7YlmP6mOJTbg3ywhueeloIndVmiXQYxYPV_9RvI-XWEfhuQHTfTgiTRWhTbKToQ4utRWo8ZIabsgGqZmAgKl7sB-KglWLh2HRfMFZGKMCk6X1xH2NgWv3mqe8cttHyCnnS2a7qSHeei8QhoWKh3ScVARfXVy-X_eP0Ot27jc_-AfpQeqyVmdXYYoebXv1xvww94OIQo1xD_7uMhecHoP2fnxv5M7a4fYancd_4ZNNxRP1WdnD4pgew6xlhR_10H1YvAXRPsTk-KF89q0GdWYkp42Mr31PbVCwRzMzqsHMjAMeSEYe6Ls-7dN6C1R942l7FwApmrZFLonCFN2O7Pf3UBWvVXwU2m5DF0ODr4DuLoV_J2KQ4ZnFsKxzxisfFkjz-n45gKaEMYG03dlNVZJQHZWrQkJOkIc2TmDEP9NqTc-SSwjB3StmKIp4eoqPFCNVOCbpeSYHQs1mZDqFOO9uOQg1hGnyVTTqM_qqgR69zSUkwHkAJ9G2_5bwk6BjKsvn3wLYXBVZP9IuVtqM8Of5hrkXQC3LPQP-XKpIt6pHp--8oYCeNRhPAaPq50ZmUWUrZ9Do3XDLQIyTCNnn2sBm65VaJXqxXgZ4yvubWSszOjWGCTXawU4tTR2hQkvSdDIP2hZSVVb8LwJmzykx5zlgGWL6NiuRd67arb-HX8uc6JzR5p90ActyRl5OXEcyboUqC5LXiWeoDfWMGHRO7aPKnSNkG66L-yMEqKQPO0z3RA1yIM742S7bzMUPw-_72wgjhRk7KyCR2VkiYUHbeV6oTpJVp2NUQNk

In [20]:
print(ask_tax("John Maden is a foreign worker (no green card) who worked three months as a contractor and made $100,000"))

[{'id': 'rs_0115f04948d53bb8006aee1e449ba887d1a0b35a663d5a86fc', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABq7h5I2-znfTyKAbRKspl7Ax-Bjox1stHeHYHZkQ0Hp9MtPLn33kZ3o2hHb_Tp0RIX2GC3G7oGhqH0hd1gRTOsGo8fzGxd23Sqa2DVc666JzxIwDRLGdf59qXRcgXa4IXQWlA501xVomTF9UL4MjyHrbDWpJDmTGCK96o3xmDk8xOnaRT05mhpa4AM9soYuF5t3JoCe1Yn9gUhiNiRBL7WhrWb9pTfZYVIs2U93vgwxLaMhvYUeb3lQskHFGy-YeZDnON3MwDNSfHNUmgpj9LyjIYQAig0qx4LPHLdduxAXuKe_PQ-EWpKgMoS99Zgg4bmHWYcsNwEPNQ_3if-lJxfQRsLK_buss3JsJ3Ffqn1UnPz2M1ZriZ2QEMhFfKHAHi8TzR6QHNzq4fRgBgq270Cfa9XkesQVwMT_Nl28IhOr3XTZmu97edWRtC8cWpdDmKQS0Phsn2U8ui9GYLoi8w8B8oNeqpkqc9SFzLfxXeXqq46XBJB8FeFscwsHQajXbIA313ebefNgJdN8FK_306kHQ1c_JweqVAPCSxiH2RZUTo2C61tLJUIjXC2qFZBeIaBLX2jrs4JfDE5ZyCf7cB8p9YNGEIfA_GbdAFPxcbtZAeUp8OUxJlWlfW0iif8VRnD0hOjw6e111B4sYHtu4etSKUZ3FW8Yf4iniiOMmGiFHKaqHsNLlRNG4TRZcq6BrASMg_HgZxd6NggNkDymdgnw8JJHVyl9MbKq2bMU5ls3o5r9sEQBWO6MXCc8yWa4oDzZmi6qYse0jacLBPebpzC0u0lj6PQcUaSPCK85OA7Zg_lEedbh81NtGiD09obXxDNg6brqAJABjlPbAfHJCei_B

In [21]:
print(ask_tax("name='John Maden' salary='100000' status='nonresident'"))

-----> CALCULATING TAX NON RESIDENT ON INCOME 100000.0
[{'type': 'text', 'text': 'John Maden is listed as a nonresident. For a salary of $100,000, the estimated tax liability is **$25,000**.', 'annotations': [], 'id': 'msg_0302f559fb917fe8006aee1e5207a487d1a32086d63d215610', 'phase': 'final_answer'}]


# DataFrame agent
https://python.langchain.com/docs/integrations/toolkits/pandas

In [22]:
import langchain, langchain_experimental
print(langchain.__version__)
print(langchain_experimental.__version__)

1.4.9
0.4.2


/var/folders/h4/gc9911bs2sq4x08kxr_589rw0000gp/T/ipykernel_26929/282852513.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  import langchain, langchain_experimental


In [23]:
import pandas as pd
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI

In [24]:
df = pd.read_csv("data/mpg.csv")
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


In [25]:
agent2 = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    allow_dangerous_code=True,
    agent_type="tool-calling",
)

In [26]:
agent2.invoke("how many rows are there?")



> Entering new AgentExecutor chain...
[{'id': 'rs_043cafd0481e90a2006aee1e62e93c87d1b6686f1bd446e29a', 'summary': [], 'type': 'reasoning', 'content': [], 'index': 0, 'encrypted_content': 'gAAAAABq7h5jg0xk0i9RM3ckIqz3haw-DhQRxTrg25_GKFV3XVq_QSiUTBfhkK-wksDxacC2XPMO7O6usMXwNZ9KmrTWi3P2QD9vs8kZbCZBAYAk5gz6-6aOE8N7_ImnlvfV6ifCxRNqMZQmrxmCeX29MRUFVhxJ8dFD9HVHqQRGL1IO3l4rH_wBzKXgJx8WQ-gUJWmDXOVXwWnoZh77HTwPOFWzmAmNT3Tj22yyT8uZUS2Rq3SoyjVGEjmOgNnYqUfNC7bJAPYPNvXLnsJ1c21xRnypFRK_e3RB8YAb_UFZEFLZcAzVCeMwxCXjsW0u81mN7eAtlPJIDif7T1YMHCrZUqv0WxbNpp9I11UFQcBzWyHy6wWA5bQsIUo6e29_NLIRyD-nTSJFLAU3TRYRDnKjRCBPSO6PU12meCvwuo4xtF3yLJ5zdHfzv9sVx23RecmebDlRFkgvvDZ3eRK_Vz9OwssloVIrVXFI_Di_4y4CSYPCDq5UkMNEWH_gQPim7AaI7zzv3oS8Fw3R4emhIeA8rUtkJ8j88gvHcxSaz-QdKG1xi0j04spkKSMxdpvU3TVWoil1uhNkVMxoRSA8-P4QJdQ3rL97FU8GI2qeG9nW0AJn9dIlQXGnpKVIOWGOEf-R4R-s6jExU8wc1BrJmMSsPYVXQR10F5dnL7DxRz0bqcMHwHJ7lNxp4dJZcD5SsfLt24ycG3UseUlspaSmjbY0Y820DZmZMKq2Zbgdd3yw_SIjI-ffDjNrw9NK88gXbccJaURuea8RkMtSHhSe9m60EtnmxPA7-_Tn7soDFh

{'input': 'how many rows are there?',
 'output': [{'id': 'rs_043cafd0481e90a2006aee1e62e93c87d1b6686f1bd446e29a',
   'summary': [],
   'type': 'reasoning',
   'content': [],
   'index': 0,
   'encrypted_content': 'gAAAAABq7h5jg0xk0i9RM3ckIqz3haw-DhQRxTrg25_GKFV3XVq_QSiUTBfhkK-wksDxacC2XPMO7O6usMXwNZ9KmrTWi3P2QD9vs8kZbCZBAYAk5gz6-6aOE8N7_ImnlvfV6ifCxRNqMZQmrxmCeX29MRUFVhxJ8dFD9HVHqQRGL1IO3l4rH_wBzKXgJx8WQ-gUJWmDXOVXwWnoZh77HTwPOFWzmAmNT3Tj22yyT8uZUS2Rq3SoyjVGEjmOgNnYqUfNC7bJAPYPNvXLnsJ1c21xRnypFRK_e3RB8YAb_UFZEFLZcAzVCeMwxCXjsW0u81mN7eAtlPJIDif7T1YMHCrZUqv0WxbNpp9I11UFQcBzWyHy6wWA5bQsIUo6e29_NLIRyD-nTSJFLAU3TRYRDnKjRCBPSO6PU12meCvwuo4xtF3yLJ5zdHfzv9sVx23RecmebDlRFkgvvDZ3eRK_Vz9OwssloVIrVXFI_Di_4y4CSYPCDq5UkMNEWH_gQPim7AaI7zzv3oS8Fw3R4emhIeA8rUtkJ8j88gvHcxSaz-QdKG1xi0j04spkKSMxdpvU3TVWoil1uhNkVMxoRSA8-P4QJdQ3rL97FU8GI2qeG9nW0AJn9dIlQXGnpKVIOWGOEf-R4R-s6jExU8wc1BrJmMSsPYVXQR10F5dnL7DxRz0bqcMHwHJ7lNxp4dJZcD5SsfLt24ycG3UseUlspaSmjbY0Y820DZmZMKq2Zbgdd3yw_SIjI-ffDjNrw9NK88gXbccJaURuea8RkMtSHh

In [27]:
agent2.invoke("how many rows are there?",)



> Entering new AgentExecutor chain...
[{'id': 'rs_0625dc075ea9c217006aee1e650f8087d1b0e1613ab1b176a8', 'summary': [], 'type': 'reasoning', 'content': [], 'index': 0, 'encrypted_content': 'gAAAAABq7h5mQBZSuUzBrULRju2e7SL4nzMKcEVcSl2ZpbEmwLOS7BxaxNZY79lhYOe1XkvWiYadyxydElckSwXbE2Gh-lPJ8-YFo2nUTl8JFK4ZTI610FuT-KEEAXDSJ40DKZOf6Y0EpV75bzU6OViwA1w5SuK7FG9kBV7ijR5XBgMn3k0JC88Z-1SnO7UmYyp7159rn-lIoy7Ike9DMMCMMv5MgS-rSmxQIMEwsfoJpLQIX_00ZlYalO5yNyHRC2YDjRuQ0weD12Jxi7nDnubXPpFlUrBsYzI7MpVANAQJH1wV774Y0Mg4x6OAPU3hQ1nvdCE9NyDTxlUqWeK6WzamMeiqwRn6lXeFCAd7GFUBCWgYGrgSGTbPD-wujaRBq_TXJ0JD4205us8U-LjwCV5EoKacht5lsyhiL8nmGtwgb9eNvm6oBUmmdAOYLvzQm7XV7a9daUgnvsJVeQjDFyH7zTpo2fpMsxSmcdzIdGFH_PSOf1G4RZvm8SSHnHVrNzCN6hRItfaxH18GsDC5xkq6whUBVMIpBmzkwqqq2NOEVYODyf7-QpuMImtmt6hBXRE9dFWY3alPcc8AwGI7q030aOkRltjyLWYY78O9c2AZLgluqCAlMfsOh224i4P_sKvrgWFk4UJUi4V_rH12wN6rA62TWxqUdL03hxcFzWDMnL-bZVwypEgUaDJl7_pn3dGBncR8b7PNH20wDf23xyXN1SCtTyAqwixVLaukFMea0zyw1eVVKg_eAjvo_BqXVenE9CfKYf1rks3XLAzM5DOI8MzCSHQt78rIYsOKL2

{'input': 'how many rows are there?',
 'output': [{'id': 'rs_0625dc075ea9c217006aee1e650f8087d1b0e1613ab1b176a8',
   'summary': [],
   'type': 'reasoning',
   'content': [],
   'index': 0,
   'encrypted_content': 'gAAAAABq7h5mQBZSuUzBrULRju2e7SL4nzMKcEVcSl2ZpbEmwLOS7BxaxNZY79lhYOe1XkvWiYadyxydElckSwXbE2Gh-lPJ8-YFo2nUTl8JFK4ZTI610FuT-KEEAXDSJ40DKZOf6Y0EpV75bzU6OViwA1w5SuK7FG9kBV7ijR5XBgMn3k0JC88Z-1SnO7UmYyp7159rn-lIoy7Ike9DMMCMMv5MgS-rSmxQIMEwsfoJpLQIX_00ZlYalO5yNyHRC2YDjRuQ0weD12Jxi7nDnubXPpFlUrBsYzI7MpVANAQJH1wV774Y0Mg4x6OAPU3hQ1nvdCE9NyDTxlUqWeK6WzamMeiqwRn6lXeFCAd7GFUBCWgYGrgSGTbPD-wujaRBq_TXJ0JD4205us8U-LjwCV5EoKacht5lsyhiL8nmGtwgb9eNvm6oBUmmdAOYLvzQm7XV7a9daUgnvsJVeQjDFyH7zTpo2fpMsxSmcdzIdGFH_PSOf1G4RZvm8SSHnHVrNzCN6hRItfaxH18GsDC5xkq6whUBVMIpBmzkwqqq2NOEVYODyf7-QpuMImtmt6hBXRE9dFWY3alPcc8AwGI7q030aOkRltjyLWYY78O9c2AZLgluqCAlMfsOh224i4P_sKvrgWFk4UJUi4V_rH12wN6rA62TWxqUdL03hxcFzWDMnL-bZVwypEgUaDJl7_pn3dGBncR8b7PNH20wDf23xyXN1SCtTyAqwixVLaukFMea0zyw1eVVKg_eAjvo_BqXVenE9CfKYf1rks3XLA

In [28]:
agent2.run("what country has the highest average mpg?")

/var/folders/h4/gc9911bs2sq4x08kxr_589rw0000gp/T/ipykernel_26929/1269757213.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  agent2.run("what country has the highest average mpg?")




> Entering new AgentExecutor chain...
[{'id': 'rs_08504b91410dfd84006aee1e7968c487d19490794ad9defd98', 'summary': [], 'type': 'reasoning', 'content': [], 'index': 0, 'encrypted_content': 'gAAAAABq7h55T4_j4dg0pFfBb9W__ZGUFWxfAEBhRz-nkGvv2znkaRlCOqcB39_7qvkXMpLcuK9UP0CmpzJD3otqvMJL9puzw6hent3y5530IITIys7QdcDR3rwLNwDEQXj39lcQjoTMjJgr8X-yseO3xGUDSmYzZCdkHArD4qeLnEl5IT7qhD1MWqQpATMer1lVOcWjN5Meui8Z58gC7cJ5J2BQ9FhHSxZFR3rXfUfBqWpbN-oGIaHNjU0sfoED9mLm2pWTehpHYRUOmWIRzh7S-vTS5qfFS1eliYOhppr0LQ9Dzt7D-Z3VoqrKp-pCefhpw_K5qTY-GzPNbQkXbGOrNfGlGLnjW26ZUKHutQ0t8tHgFBhoFSHqEUeCQ5OVDJfItNFjgB81jh3891a7YY4B1fhGah9Mlc3o14Iz7PJeWdDKFkCwMZfr03OOVRXFJjo6x6kpEdybTLuX67QI0j_NSyWewUdz9w3Ta4XgEOT3xDJFqjE4u18ixVCFIb1b0gyyNc62Z_BqsM2rZt3oZT6QcSF9qmsIBOKo6OMrefsm0Q_jajuX0E6sR-AxlJXg5Mhcx6qFPRi2Xvi4HtMvgAYj3ePDCjl3_wikh3OxVpyI56L0aXsu-A9hpkKxK_Z25v98Ef6r8BKzrP6pAQFPUFJoVzDI8Z8kOcrqo2e1DUUG2VBu69ShMOrh7FfL9jXKZgSxAaJ_hH5RGo2bjrO9usV7G9_P1-wpPQhdml0HC4lM4XxYglCJ0LnKyXmyNcBLTpaqolCLK01Yksp1Fq2oON32aDOgbKlDeyf6ozYszP

[{'id': 'rs_08504b91410dfd84006aee1e7968c487d19490794ad9defd98',
  'summary': [],
  'type': 'reasoning',
  'content': [],
  'index': 0,
  'encrypted_content': 'gAAAAABq7h55T4_j4dg0pFfBb9W__ZGUFWxfAEBhRz-nkGvv2znkaRlCOqcB39_7qvkXMpLcuK9UP0CmpzJD3otqvMJL9puzw6hent3y5530IITIys7QdcDR3rwLNwDEQXj39lcQjoTMjJgr8X-yseO3xGUDSmYzZCdkHArD4qeLnEl5IT7qhD1MWqQpATMer1lVOcWjN5Meui8Z58gC7cJ5J2BQ9FhHSxZFR3rXfUfBqWpbN-oGIaHNjU0sfoED9mLm2pWTehpHYRUOmWIRzh7S-vTS5qfFS1eliYOhppr0LQ9Dzt7D-Z3VoqrKp-pCefhpw_K5qTY-GzPNbQkXbGOrNfGlGLnjW26ZUKHutQ0t8tHgFBhoFSHqEUeCQ5OVDJfItNFjgB81jh3891a7YY4B1fhGah9Mlc3o14Iz7PJeWdDKFkCwMZfr03OOVRXFJjo6x6kpEdybTLuX67QI0j_NSyWewUdz9w3Ta4XgEOT3xDJFqjE4u18ixVCFIb1b0gyyNc62Z_BqsM2rZt3oZT6QcSF9qmsIBOKo6OMrefsm0Q_jajuX0E6sR-AxlJXg5Mhcx6qFPRi2Xvi4HtMvgAYj3ePDCjl3_wikh3OxVpyI56L0aXsu-A9hpkKxK_Z25v98Ef6r8BKzrP6pAQFPUFJoVzDI8Z8kOcrqo2e1DUUG2VBu69ShMOrh7FfL9jXKZgSxAaJ_hH5RGo2bjrO9usV7G9_P1-wpPQhdml0HC4lM4XxYglCJ0LnKyXmyNcBLTpaqolCLK01Yksp1Fq2oON32aDOgbKlDeyf6ozYszPPy2ojX2iprLO5HrSv7kxgFcrQGxDoq

In [29]:
agent2.run("what country has the highest average mpg?",)



> Entering new AgentExecutor chain...
[{'id': 'rs_036347dbb2ba0e4e006aee1e7b723887d1adb9faca26440e77', 'summary': [], 'type': 'reasoning', 'content': [], 'index': 0, 'encrypted_content': 'gAAAAABq7h57RelF7HrHrISJm8xIdcBUU-AC5vL5vBS07mXTlEZsYqLBdYtFHw2JS4aviytl_6d2WBjcYY5fMIWpp3fpTdnf9Xkw1wUsQ4O6JYaBLU9aNHCECa2WIES27MlWjL0LOEnI4Um9yyN9IWH69ptvGpLK5NxkjgfGHlKgkk1ocqLNTTw3y7EYrhyhz7eDhmyl2_fQWyuP4EjDqkn6C_VxXOOLqlzWINC-vjQ0zacGrKR26WrgONu00fwULbZHCVgkgcIZWMjmnN6zxxfLq-Y9mEbFa0U8rIDSjok5Hu0J6tkkmosnigWuD6i1xe6fPOAE-UfaNpFWpW2VtkOWgOHSdtzt8lldtKITwqnocIgzp48jjgTPNVvAvwa4uFXN0DypdsO-8NotsOCGzRqUcNZgBeBX-VR2vzGxAuLnfiH_eckiXMlc7lODccUR6_4GgKtmOmkrPyiDln1rZI9Pk3O1x9cmJ8OB7baIoJD8OdwmpIuzLycpLES73sV3X5ee9ZdeQhJz0weI3Qn1CklymArxwsClZAt1wRrS0cdxm2ZC5fXphMhkd-oxDuYBcykJJ1Z4csM5GKfTqz5Ofz9NPnd7q47HPdoCe-RKO5hT-G8wd9pUBo17Hgy_hgZK3SrjrF0qfrsQQMKPTiX0RmnG_WtSWq7_RQuxEBG-vrYidIuMkFHuDsFicEO2g-yJyBFsjfz8KeC_f6rMk6T7Bt12amSRnQoy8XCdftsQIRc1unCkxsKwcOxHOQNTS4T0RVAjgZWz13L4fKstWZeh00eA8xaFYDkwk0RpZjRdds

[{'id': 'rs_036347dbb2ba0e4e006aee1e7b723887d1adb9faca26440e77',
  'summary': [],
  'type': 'reasoning',
  'content': [],
  'index': 0,
  'encrypted_content': 'gAAAAABq7h57RelF7HrHrISJm8xIdcBUU-AC5vL5vBS07mXTlEZsYqLBdYtFHw2JS4aviytl_6d2WBjcYY5fMIWpp3fpTdnf9Xkw1wUsQ4O6JYaBLU9aNHCECa2WIES27MlWjL0LOEnI4Um9yyN9IWH69ptvGpLK5NxkjgfGHlKgkk1ocqLNTTw3y7EYrhyhz7eDhmyl2_fQWyuP4EjDqkn6C_VxXOOLqlzWINC-vjQ0zacGrKR26WrgONu00fwULbZHCVgkgcIZWMjmnN6zxxfLq-Y9mEbFa0U8rIDSjok5Hu0J6tkkmosnigWuD6i1xe6fPOAE-UfaNpFWpW2VtkOWgOHSdtzt8lldtKITwqnocIgzp48jjgTPNVvAvwa4uFXN0DypdsO-8NotsOCGzRqUcNZgBeBX-VR2vzGxAuLnfiH_eckiXMlc7lODccUR6_4GgKtmOmkrPyiDln1rZI9Pk3O1x9cmJ8OB7baIoJD8OdwmpIuzLycpLES73sV3X5ee9ZdeQhJz0weI3Qn1CklymArxwsClZAt1wRrS0cdxm2ZC5fXphMhkd-oxDuYBcykJJ1Z4csM5GKfTqz5Ofz9NPnd7q47HPdoCe-RKO5hT-G8wd9pUBo17Hgy_hgZK3SrjrF0qfrsQQMKPTiX0RmnG_WtSWq7_RQuxEBG-vrYidIuMkFHuDsFicEO2g-yJyBFsjfz8KeC_f6rMk6T7Bt12amSRnQoy8XCdftsQIRc1unCkxsKwcOxHOQNTS4T0RVAjgZWz13L4fKstWZeh00eA8xaFYDkwk0RpZjRddsC6LQDvdNd_IVksVW-mok26te3CuMDc

# SQL Database

https://python.langchain.com/docs/integrations/toolkits/sql_database

In [30]:
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.agents import create_agent

In [31]:
db = SQLDatabase.from_uri("sqlite:///data/ClassicModels.db")
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

/opt/anaconda3/lib/python3.14/site-packages/langchain_community/utilities/sql_database.py:164: SAWarning: Could not instantiate type <class 'sqlalchemy.sql.sqltypes.INTEGER'> with reflected arguments ['11']; using no arguments.
  self._metadata.reflect(
/opt/anaconda3/lib/python3.14/site-packages/langchain_community/utilities/sql_database.py:164: SAWarning: Could not instantiate type <class 'sqlalchemy.sql.sqltypes.SMALLINT'> with reflected arguments ['6']; using no arguments.
  self._metadata.reflect(


In [32]:
sql_tools = toolkit.get_tools()
sql_tools

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x16fac5be0>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x16fac5be0>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x16fac5be0>),
 QuerySQLCheckerTool(description='Use this tool to double check if your 

In [33]:
sql_agent = create_agent(
    model=llm,
    tools=sql_tools,
    system_prompt=(
        "You are an agent designed to interact with a SQL database. "
        "Use the provided tools to inspect schema and run queries safely."
    ),
)

In [34]:
resp = sql_agent.invoke({"messages": [{"role": "user", "content": "List the tables in the database."}]})
print(resp["messages"][-1].content)

[{'type': 'text', 'text': 'The database contains these tables:\n\n- Customers\n- Employees\n- Offices\n- OrderDetails\n- Orders\n- Payments\n- ProductLines\n- Products', 'annotations': [], 'id': 'msg_0757619fb691547a006aee1e8e708487d19cf55e2f947876de', 'phase': 'final_answer'}]


In [35]:
resp

{'messages': [HumanMessage(content='List the tables in the database.', additional_kwargs={}, response_metadata={}, id='d88ea5a2-0518-4f06-9a31-4b19a78e70cf'),
  AIMessage(content=[{'arguments': '{"tool_input":""}', 'call_id': 'call_qjp5FxxffufduQ3dLzhtye6i', 'name': 'sql_db_list_tables', 'type': 'function_call', 'id': 'fc_0757619fb691547a006aee1e8d482087d1bbf1f0ac6aefa8d7', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0757619fb691547a006aee1e8cbb3c87d1b1e0f51e254a6a9f', 'created_at': 1793990284.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0757619fb691547a006aee1e8cbb3c87d1b1e0f51e254a6a9f', tool_calls=[{'name': 'sql_db_list_tables', 'args': {'tool_input': ''}, 'id': 'call_qjp5FxxffufduQ3dLzhtye6i', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 352, 'output_tokens': 21, 'total_tokens': 373,

In [36]:
resp = sql_agent.invoke({"messages": [{"role": "user", "content": "Select the first five rows in Employees."}]})
print(resp["messages"][-1].content)

[{'type': 'text', 'text': '| employeeNumber | lastName | firstName | extension | email | reportsTo | jobTitle | officeCode |\n|---:|---|---|---|---|---:|---|---|\n| 1002 | Murphy | Diane | x5800 | dmurphy@classicmodelcars.com | 0 | President | 1 |\n| 1056 | Patterson | Mary | x4611 | mpatterso@classicmodelcars.com | 1002 | VP Sales | 1 |\n| 1076 | Firrelli | Jeff | x9273 | jfirrelli@classicmodelcars.com | 1002 | VP Marketing | 1 |\n| 1088 | Patterson | William | x4871 | wpatterson@classicmodelcars.com | 1056 | Sales Manager (APAC) | 6 |\n| 1102 | Bondur | Gerard | x5408 | gbondur@classicmodelcars.com | 1056 | Sale Manager (EMEA) | 4 |', 'annotations': [], 'id': 'msg_0b77e53ceef5e0ba006aee1e9a87a487d186cb8c40fee75c76', 'phase': 'final_answer'}]


In [37]:
resp

{'messages': [HumanMessage(content='Select the first five rows in Employees.', additional_kwargs={}, response_metadata={}, id='4b431c7f-d290-4383-ab35-e9901ccfef11'),
  AIMessage(content=[{'id': 'rs_0b77e53ceef5e0ba006aee1e90bd1087d18f16618536e79df6', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABq7h6R5yV1fpcmvqAV4tCw_BqSmG93Pi47yL7ZZwrU1MjJsClvFNeBOlQRwfls5V0Um_RpDvb8YeLbIejKQ_3LMhbOndbAimpuU0HrIHJaeNClXMclMYLpRvn1OIIRjoDtnRbFnF_Rums4XYUEsx9qu8Dsb0_SHpQ1a5aYwi-2p7trAILe-ddbg1y6dH6IcgaR9eqYqPQPKYMoqyJtfzcaLRH9RHtqTSFARlymr3sVhb1M1NuYaXuHTiGHPh8vwiMnCmcEdb1jmAjpe4b91Z6n0202DlsJndbhDFHIXgYaxpPU5Uq05oY6Gk67LpzfuOhxLxWEnm8Kt317rHdaXO8CtWQxZVu16ibvP6T_GANcUBw9nJx0JCW2wbsXVzSDVvnufVGj5e5l7MIT-Yeh0Xc2kb1vHyiYV8aY5U72M0mlZQpcnlFxziGuXuIfuDnjPE9pNmOcM68pHQfRZY2jDXkDy1_AL5puZq4325PkdNfKdrFTElIIfgCHJwnvqkcnt36jKhYxhQ10OB_dKAHN_bNiAljxfLw9YdnKHRmtgyRJgIpBiEp2DeIIyuX4TOVZ099riLkvvtIvY6H-MyPfQdXfOswVJ2NxVc4HMc5inCayjXc_3PlZTE7KjOOuQbOKajq81jodtevJSF6SBhhb0SFGR7dn79m